# Геокодирование через Nominatim (OSM) для сайта ГВС-Ярославль

Скрипт загружает адреса из репозитория, геокодирует их через OpenStreetMap Nominatim и сохраняет результат в `geo.json`.

In [ ]:
# ШАГ 1 - Установка библиотек
!pip install requests --quiet
print('✅ Готово')

In [ ]:
# ШАГ 2 - Импорты и настройки
import requests, json, time, os, re
from google.colab import files

GEO_FILE = "geo.json"
DELAY    = 1.1

NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
HEADERS = {"User-Agent": "gvs-yaroslavl/1.0 (github.com/Aoneti/gvs-yaroslavl)"}

print("✓ Импорты и настройки загружены")

In [ ]:
# ШАГ 3 - Географические границы
YAROSLAVL_BBOX = {
    "lat_min": 57.52,
    "lat_max": 57.73,
    "lng_min": 39.74,
    "lng_max": 40.08,
}

# Границы для сельских поселений
YAROSLAVL_DISTRICT_BBOX = {
    "lat_min": 57.30,
    "lat_max": 57.90,
    "lng_min": 39.40,
    "lng_max": 40.40,
}

# Сёла и посёлки вне Ярославля
RURAL_SETTLEMENTS = {
    "дубки":         "Карабихское сельское поселение, Ярославский район",
    "щедрино":       "Карабихское сельское поселение, Ярославский район",
    "карабиха":      "Карабихское сельское поселение, Ярославский район",
    "толбухино":     "Толбухинское сельское поселение, Ярославский район",
    "лесная поляна": "Ярославский район",
}

def get_locality(addr_lower: str) -> str:
    for keyword, locality in RURAL_SETTLEMENTS.items():
        if keyword in addr_lower:
            return locality
    return "Ярославль"

def is_rural(addr_lower: str) -> bool:
    return any(k in addr_lower for k in RURAL_SETTLEMENTS)

print("✓ Географические границы и справочник загружены")

In [ ]:
# ШАГ 4 - Функции и валидации
def is_valid_location(lat: float, lng: float, rural: bool) -> bool:
    """Проверяем, что точка действительно в Ярославле (или районе)."""
    bbox = YAROSLAVL_DISTRICT_BBOX if rural else YAROSLAVL_BBOX
    return (bbox["lat_min"] <= lat <= bbox["lat_max"] and
            bbox["lng_min"] <= lng <= bbox["lng_max"])

def display_name_ok(display_name: str, rural: bool) -> bool:
    """
    Проверяем display_name из Nominatim.
    Должно содержать 'Ярославль' или 'Ярославский район'.
    """
    dn = display_name.lower()
    if rural:
        return "ярославский район" in dn or "ярославская область" in dn
    return "ярославль" in dn and "ярославский район" not in dn

def validate_result(result: dict, addr_lower: str) -> bool:
    """Два независимых теста — оба должны пройти."""
    rural = is_rural(addr_lower)
    lat, lng = result["lat"], result["lng"]
    dn  = result.get("full_name", "")

    coord_ok = is_valid_location(lat, lng, rural)
    name_ok  = display_name_ok(dn, rural)

    if not coord_ok:
        print(f"      ⛔ bbox fail: {lat:.4f}, {lng:.4f} — {dn[:60]}")
    elif not name_ok:
        print(f"      ⛔ name fail: {dn[:80]}")

    return coord_ok and name_ok

print("✓ Функции валидации загружены")

In [ ]:
# ШАГ 5 - Нормализация адресов
def normalize_address(raw: str) -> str:
    s = raw.strip()

    # 1. Порядковые числительные: 2-ой, 1-ый, 3-ей → 2-й, 1-й, 3-й
    s = re.sub(r"\b(\d+)-(ой|ый|ий|ей)\b", r"\1-й", s, flags=re.IGNORECASE)
    s = re.sub(r"\b(\d+)-(ая|яя)\b",        r"\1-я", s, flags=re.IGNORECASE)
    s = re.sub(r"\b(\d+)-(ое|ее)\b",        r"\1-е", s, flags=re.IGNORECASE)

    # 2. Корпус → формат Nominatim «NNNкM»
    s = re.sub(
        r"д\.?\s*(\d+\w*)[,\s]+к(?:орп?)?\.?\s*(\d+)",
        r"\1к\2", s, flags=re.IGNORECASE
    )
    s = re.sub(r"\bк(?:орп?)?\.?\s*(\d+)", r"к\1", s, flags=re.IGNORECASE)

    # 3. Типы улиц (порядок важен: пр-д до пр)
    street_types = [
        (r"\bпр-кт\.?\b",         "проспект"),
        (r"\bпр-д\.?\b",          "проезд"),
        (r"\bпер\.?\b",           "переулок"),
        (r"\bпл\.?\b",            "площадь"),
        (r"\bб-р\.?\b",           "бульвар"),
        (r"\bнаб\.?\b",           "набережная"),
        (r"\bмкр\.?\b",           "микрорайон"),
        (r"\bкв-л\.?\b",          "квартал"),
        (r"\bш\.?\b",             "шоссе"),
        (r"\bул\.?\b",            "улица"),
        (r"\bпр\.(?!\s*-)",       "проспект"),
    ]
    for pattern, replacement in street_types:
        s = re.sub(pattern, replacement, s, flags=re.IGNORECASE)

    # 4. Убираем остаточные точки после типа улицы
    s = re.sub(r"\bулица\.",    "улица",    s)
    s = re.sub(r"\bпроспект\.", "проспект", s)

    # 5. Убираем «д.» перед числами
    s = re.sub(r"\bд\.?\s*(?=\d)", "", s, flags=re.IGNORECASE)

    # 6. Строение и литера
    s = re.sub(r"\bстр\.?\s*(\d+)", r"с\1", s, flags=re.IGNORECASE)
    s = re.sub(r"\bлит\.?\s*\w*",   "",     s, flags=re.IGNORECASE)

    # 7. Чистим пробелы
    s = re.sub(r"\s{2,}", " ", s).strip(" ,.")
    return s

# Проверка нормализации
test_cases = [
    "2-ой Брагинский пр-д д. 3",
    "пр-кт Авиаторов, д.74",
    "ул Красноборская, д.38, корп.3",
    "ул. Шоссейная 1-я, д.32, корп.2",
    "ул. Пушкина, д.20",
    "пер Первомайский, д.5",
]
print("Проверка нормализации:")
for t in test_cases:
    print(f"  {t}  →  {normalize_address(t)}")

In [ ]:
# ШАГ 6 - Функция геокодирования
def geocode_address(addr: str) -> dict | None:
    norm     = normalize_address(addr)
    locality = get_locality(addr.lower())
    rural    = is_rural(addr.lower())
    street_only = re.sub(r"[,\s]+\d+\S*\s*$", "", norm).strip()

    strategies = [
        # 1. Структурированный + нормализованный (самый точный)
        {"street": norm, "city": locality, "country": "Россия",
         "format": "jsonv2", "limit": 3},
        # 2. Свободный + нормализованный
        {"q": f"{locality}, {norm}",  "format": "jsonv2", "limit": 3},
        # 3. Свободный + оригинальный
        {"q": f"{locality}, {addr}",  "format": "jsonv2", "limit": 3},
        # 4. Только улица — хотя бы приблизительные координаты
        {"q": f"{locality}, {street_only}", "format": "jsonv2", "limit": 3},
    ]

    for i, params in enumerate(strategies):
        if i > 0:
            time.sleep(DELAY)
        try:
            r = requests.get(NOMINATIM_URL, params=params, headers=HEADERS, timeout=10)
            if r.status_code != 200:
                continue
            candidates = r.json()
            if not candidates:
                continue

            for candidate in candidates:
                lat = round(float(candidate["lat"]), 6)
                lng = round(float(candidate["lon"]), 6)
                dn  = candidate.get("display_name", "")
                result = {"lat": lat, "lng": lng, "full_name": dn,
                          "strategy": i + 1, "norm": norm}
                if validate_result(result, addr.lower()):
                    return result

        except Exception as e:
            print(f"    ⚠️  Стратегия {i+1}: {e}")

    return None

print("✓ Функция геокодирования загружена")

In [ ]:
# ШАГ 7 - Загрузка данных из репозитория
print("=" * 58)
print("Загрузка data.json с GitHub...")
url = "https://raw.githubusercontent.com/Aoneti/gvs-yaroslavl/refs/heads/main/data.json"
resp = requests.get(url, timeout=15)
print(f"  HTTP статус: {resp.status_code}")
data = resp.json()
print(f"  Записей: {len(data)}")

addresses = sorted(set(item["address"] for item in data))
print(f"  Уникальных адресов: {len(addresses)}")

In [ ]:
# ШАГ 7.1 (опционально) - загрузить существующий geo.json для продолжения работы, если вы уже запускали скрипт раньше и хотите продолжить с того же места
# Раскомментируйте строки ниже, если хотите загрузить кэш вручную!!!

# uploaded = files.upload()
# print("Файл загружен:", list(uploaded.keys()))

print("ℹ️  Ячейка для загрузки кэша (опционально). Раскомментируйте при необходимости.")

In [ ]:
# ШАГ 8 - Инициализация кэша и очередь обработки
geo_db = {}
if os.path.exists(GEO_FILE):
    with open(GEO_FILE, encoding="utf-8") as f:
        geo_db = json.load(f)
    done = sum(1 for v in geo_db.values() if v)
    print(f"  ♻️  Кэш: {len(geo_db)} записей, {done} с координатами")

    # Сбрасываем None — повторим с новой валидацией
    failed = [k for k, v in geo_db.items() if v is None]
    for k in failed:
        del geo_db[k]
    if failed:
        print(f"  🔄 Сброшено {len(failed)} неудачных — повторим")

    # Сбрасываем результаты вне bbox
    invalid = []
    for k, v in list(geo_db.items()):
        if v and not is_valid_location(v["lat"], v["lng"], is_rural(k.lower())):
            invalid.append(k)
            del geo_db[k]
    if invalid:
        print(f"  🔄 Сброшено {len(invalid)} адресов вне bbox — перегеокодируем:")
        for a in invalid:
            print(f"     • {a}")
else:
    print("  Кэш не найден, начинаем с нуля")

to_process = [a for a in addresses if a not in geo_db]
print(f"  Осталось обработать: {len(to_process)}")
print(f"  Ожидаемое время: ~{len(to_process) * DELAY * 2 / 60:.0f} мин")

In [ ]:
# ШАГ 9 - Геокодирование
# ⚠️ Не прерывайте выполнение — кэш сохраняется после каждого адреса. Если сессия прервётся, скачайте geo.json и загрузите его в следующий раз через ШАГ 7.1
print("Геокодирование...")
print("-" * 58)

ok   = sum(1 for v in geo_db.values() if v)
fail = 0

for i, addr in enumerate(to_process):
    result = geocode_address(addr)

    if result:
        geo_db[addr] = result
        ok += 1
        strat  = result["strategy"]
        marker = "✅" if strat == 1 else ("🔶" if strat <= 3 else "🔸")
        print(f"  [{i+1}/{len(to_process)}] {marker}(стр.{strat}) {addr[:38]}  →  {result['norm']}")
    else:
        geo_db[addr] = None
        fail += 1
        print(f"  [{i+1}/{len(to_process)}] ❌  {addr}  →  {normalize_address(addr)}")

    # Сохраняем кэш после каждого адреса
    with open(GEO_FILE, "w", encoding="utf-8") as f:
        json.dump(geo_db, f, ensure_ascii=False, indent=2)

    time.sleep(DELAY)

print("\n" + "=" * 58)
print(f"✅ Успешно: {ok} из {len(addresses)}")
if fail:
    print(f"❌ Не найдено: {fail}")
    not_found = [a for a, v in geo_db.items() if not v]
    for a in not_found:
        print(f"    • {a}  →  {normalize_address(a)}")

s = [0, 0, 0, 0]
for v in geo_db.values():
    if v and v.get("strategy"):
        idx = v["strategy"] - 1
        if 0 <= idx < 4:
            s[idx] += 1
print(f"\n  Стр.1 структурир.+норм.: {s[0]}")
print(f"  Стр.2 свободный+норм.:   {s[1]}")
print(f"  Стр.3 свободный+ориг.:   {s[2]}")
print(f"  Стр.4 только улица:      {s[3]}")

In [ ]:
# ШАГ 10 - Скачать результат
print(f"Скачиваю {GEO_FILE}…")
files.download(GEO_FILE)
print("✓ Готово! Положите geo.json рядом с index.html в репозитории.")